# La carrera de la mina

Programa un robot autónomo que llegue al tesoro antes que su rival. Las minas del torneo se generarán después de cerrar las entregas. Tu robot recibe el mapa completo de la partida y debe decidir qué hacer en cada turno.

- Los bots se mueven a la vez y pueden compartir casilla. Si llegan al tesoro en el mismo turno, empatan. Tras 60 turnos sin ninguna llegada también hay empate.
- Una casilla normal (`.`) consume 1 de energía y el terreno difícil (`:`) consume 3. Los muros (`#`) son infranqueables. Esperar no consume energía.
- El primer bot que recoge la bomba deja al otro sin moverse durante los dos turnos siguientes. Si ambos llegan a la vez, la bomba se gasta y no retrasa a nadie.

## Prueba el motor

Ejecuta la partida en directo y observa cómo cambian las posiciones, la batería y el retraso en cada turno. Cambia la semilla para probar otra mina.

🔵 Bot 1 · 🟠 Bot 2 · 🟣 Ambos bots · 💎 Tesoro · 💣 Bomba · 🧱 Muro · ⬛ Terreno normal · 🟫 Terreno difícil

In [ ]:
from pathlib import Path
import sys
from urllib.request import urlretrieve

carpeta = Path("mina")
carpeta.mkdir(exist_ok=True)
for nom in ("mina.py", "bot_model.py"):
    fitxer = carpeta / nom
    if not fitxer.exists():
        urlretrieve("https://raw.githubusercontent.com/lawer/mia/main/apunts/1.-primer_trimestre/3.-B%C3%BAsqueda%20en%20espai%20d%27estats/mina/" + nom, fitxer)
sys.path.insert(0, str(carpeta.resolve()))

from mina import dibuixa_emojis, genera_mina, juga, successors
from bot_model import decideix as bot_model

In [ ]:
from time import sleep
from IPython.display import clear_output

def partida_en_directe(mina, bots, interval=0.35):
    def mostra(torn, robots, disponible):
        clear_output(wait=True)
        print(f"Turno {torn}")
        print(dibuixa_emojis(mina, robots, disponible))
        for i, (icona, robot) in enumerate(zip(("🔵", "🟠"), robots), 1):
            print(f"{icona} Bot {i} · 🔋 {robot.energia} · ⏳ {robot.retard}")
        sleep(interval)

    guanyador, historial = juga(mina, bots, en_torn=mostra)
    print("Empate" if guanyador is None else f"Gana el bot {guanyador + 1}")
    return guanyador, historial

mina = genera_mina(3)
guanyador, historial = partida_en_directe(mina, (bot_model, bot_model))

## Programa y entrega tu bot

1. Copia [la plantilla de Python](mina/bot_model.py) e implementa `decideix(vista)`. Devuelve `N`, `S`, `E`, `O` o `ESPERA`. Las coordenadas son `(fila, columna)`, a partir de cero.
2. Consulta `vista.mina` para ver el mapa, el tesoro y la bomba. Usa `vista.jo` y `vista.rival` para consultar posición, energía y retraso. `vista.bomba_disponible` indica si aún se puede recoger. Puedes generar movimientos válidos con `successors`.
3. Prueba BFS y A* con el estado `(posición, energía restante)`. Compara los turnos y la energía de la ruta directa con los de una ruta que pase por la bomba. Evalúa el bot con varias semillas.
4. Entrega **un archivo `.py` con tu alias**, con `decideix(vista)` y las funciones que necesites. Utiliza la biblioteca estándar de Python y los componentes del módulo `mina`. El bot tiene 150 ms por decisión. Un error o una decisión fuera de tiempo implica perder la partida.

Cada pareja jugará las mismas minas que el resto, dos partidas por mina, intercambiando los puntos de partida. Una victoria suma 3 puntos y un empate, 1. Si hay igualdad de puntos, cuentan las victorias.

In [ ]:
def decideix(vista):
    return bot_model(vista)

resultat_prova, historial_prova = partida_en_directe(mina, (decideix, bot_model))